In [ ]:
code = 'DT_FS_UT'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def DT_FS_UT(bt, start_time, end_time, orderside, method, decay, fsl, ssl, om):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        end_dt_1m = end_dt + datetime.timedelta(minutes=10)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        from_candle_close = True if method == 'CC' else False
        entry_time = start_dt
        
        # check entry
        ce_open, ce_high, ce_low, ce_close, ce_decay_price, ce_decay_flag, ce_decay_time = bt.decay_check_single_leg(start_dt, end_dt, ce_scrip, decay=decay, from_candle_close=from_candle_close, orderside=orderside, with_ohlc=True)
        pe_open, pe_high, pe_low, pe_close, pe_decay_price, pe_decay_flag, pe_decay_time = bt.decay_check_single_leg(start_dt, end_dt, pe_scrip, decay=decay, from_candle_close=from_candle_close, orderside=orderside, with_ohlc=True)
        ce_decay_time = ce_decay_time if ce_decay_time else end_dt_1m
        pe_decay_time = pe_decay_time if pe_decay_time else end_dt_1m
        
        if ce_decay_time < pe_decay_time:
            first_signal, first_signal_time, first_signal_price = 'CE', ce_decay_time, ce_decay_price
            ut, ut_scrip = 'PE', pe_scrip
            fs_sl_price, fs_sl_flag, _, _, fs_sl_time, fs_pnl = bt.sl_check_single_leg(ce_decay_time, end_dt, ce_scrip, o=(None if method == 'CC' else ce_decay_price), sl=fsl, orderside=orderside, from_candle_close=from_candle_close)
        
        elif pe_decay_time < ce_decay_time:
            ut, ut_scrip = 'CE', ce_scrip
            first_signal, first_signal_time, first_signal_price = 'PE', pe_decay_time, pe_decay_price
            fs_sl_price, fs_sl_flag, _, _, fs_sl_time, fs_pnl = bt.sl_check_single_leg(pe_decay_time, end_dt, pe_scrip, o=(None if method == 'CC' else pe_decay_price), sl=fsl, orderside=orderside, from_candle_close=from_candle_close)
        
        else:
            first_signal, first_signal_time, first_signal_price = '', '', ''
            fs_sl_price, fs_sl_flag, fs_sl_time, fs_pnl = '', False, '', 0
            ut, ut_scrip = '', ''
            
            if ce_decay_time != end_dt_1m:
                first_signal = 'BOTH'
                first_signal_time = ce_decay_time
        
        dt_entries = [ce_scrip, ce_price, pe_scrip, pe_price, first_signal, first_signal_price, first_signal_time, fs_sl_price, fs_sl_flag, fs_sl_time, fs_pnl]
        
        ut_open, ut_high, ut_low, ut_close, ut_sl_price, ut_sl_flag, ut_sl_time, ut_pnl = '', '', '', '', '', False, '', 0
        if fs_sl_time and (fs_sl_time < end_dt - datetime.timedelta(minutes=5)):
            ut_open, ut_high, ut_low, ut_close, ut_sl_price, ut_sl_flag, _, _, ut_sl_time, ut_pnl = bt.sl_check_single_leg(fs_sl_time, end_dt, ut_scrip, sl=ssl, with_ohlc=True, orderside=orderside, from_candle_close=from_candle_close)

        total_pnl = fs_pnl + ut_pnl
        ut_entries = [ut_scrip, ut_open, ut_sl_price, ut_sl_flag, ut_sl_time, ut_pnl, total_pnl]
        
        return [code, bt.index, start_time, end_time, orderside, method, decay, fsl, ssl, om, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price] + dt_entries + ut_entries
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, method, decay, fsl, ssl, om])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            
            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_Method/P_Decay/P_FSL/P_SSL/P_OM/Date/Day/DTE/EntryTime/Future/CE.Strike/CE.Open/PE.Strike/PE.Open/FirstSignal/FS.Decay.Price/FS.Decay.Time/FS.SL.Price/FS.SL.Flag/FS.SL.Time/FS.PNL')
            log_cols += ('/UT.Strike/UT.Price/UT.SL.Price/UT.SL.Flag/UT.SL.Time/UT.PNL/Total.PNL')
            log_cols = log_cols.split('/')
            
            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)
                        
                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [DT_FS_UT(bt, row.entry_time, row.exit_time, row.orderside, row.method, row.decay, row.fsl, row.ssl, row.om) for row in tqdm(chunk_parameter.itertuples(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))